## Per-source tables from `*.by-source.txt`

Two views per file:

**Hierarchical subtotals** — every row contributes to its `function`, its `bucket`, and its `phase`. So a `dev_fixed_point_dprice` row counts toward `dev_fixed_point_dprice`, toward `ded`, and toward `newton` simultaneously. Blank cells mark "this is a subtotal row" — they aren't missing values. Subtotal fractions sum the row-level `frac` from the by-source.txt (true total fraction), so subtotals can sum to < 1.0 when some runtime is unattributed (`<unknown:…>` rows or sub-threshold kernels).

**Per-source rows** — every kernel-attribution row, augmented with `phase`/`bucket`/`function`, sorted by `sum_ms`. `full_chain` dropped for compactness.

### What the labels mean

- **phase** — algorithmic phase the kernel is part of:
  - `newton` — anything inside the Newton iteration (including the final outer linear solve *and* the inner per-τ ev solve / ed / ded work). Conceptually all of this exists in service of computing one Newton step.
  - `init` — the one-shot initial price guess via `spp_price_solve` (before Newton starts).
  - `other` — unrecognised, including `<unknown:…>` rows.
- **bucket** under `newton`:
  - `inner_dp_per_newton` — the per-τ Bellman ev fixed-point solve inside `ed_ded_price_all_*`'s `sol_evs`. Detected by **any chain segment whose file is `dpsolve.fut`** (which covers `dps.poly`, `sa`, `nk`, `polyad`, and any internal helpers). Takes priority over `ded`/`ed`/`newton_dispatch` so it isn't masked by also touching e.g. `bellmanJ`.
  - `ded` — derivative-of-excess-demand work. Recognised functions: `utility_dprice_*`, `ccp_scrap_dprice_*`, `dbellman_prices_*`, `dev_fixed_point_dprice`, `dv_dprice_*`, `dccp_dprice*` / `dccp_dprices_*`, `ccp_dprice_full`, `dctp_dprice*` / `dctp_dprices_*`, `ctp_dprice_full`, `ded_dprice_*`, `ded_dprice_tau_*`, `ergodic_with_dprice`. Note: a `lup.ols` inside `ergodic_with_dprice` or a `lu.ols` inside `dev_fixed_point_dprice` is `ded`, not `ed` or `newton_linsolve`.
  - `ed` — excess-demand work: `ergodic`, `demand_supply_*`, `ed_tau*`.
  - `newton_linsolve` — the outer Newton `lu.ols` that produces `dp` (chain hits `newton`/`newton_step` but does *not* hit any `ed_ded_price_all*`).
  - `newton_dispatch` — body-level code inside `ed_ded_price_all*` that the compiler fused into kernels with no deeper named call site (e.g., the `tw`-weighted population aggregations at the end of `ed_ded_price_all`). The `function` label here is the matched `ed_ded_price_all_*` variant.
- **bucket** under `init`:
  - `init_dp` — `spp_price_solve` and its internal helpers `bellman_spp_with_deriv`, `bellman_spp_with_deriv_p`, `solve_spp_single`. (These are *spp's own* Bellman implementation, only used for the initial price guess. Dpsolve traffic during init also stays here.)
- **function** — the most-specific recognised function appearing in the chain, per a precedence list (most-specific first).

In [ ]:
import re
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

HERE = Path('.').resolve()
FUT2 = HERE.parent

# ---------------------------------------------------------------------------
# Parsing the by-source.txt format
# ---------------------------------------------------------------------------
_FILELINE_RE = re.compile(r'^(?P<file>[^:]+):(?P<lines>.+)$')

def split_chain_segment(seg):
    m = _FILELINE_RE.match(seg)
    if m:
        return m.group('file'), m.group('lines')
    return seg, ''

def shorten_path(path):
    return Path(path).name

def formatted_call_path(chain_segments):
    parts = []
    for s in chain_segments:
        f, lines = split_chain_segment(s)
        parts.append(f'{shorten_path(f)}:{lines}' if lines else shorten_path(f))
    return ' → '.join(parts)

# ---------------------------------------------------------------------------
# Function-name resolution from `def`/`entry` line ranges in the .fut files.
# def/entry may be indented (they live inside `module … = { … }` blocks).
# ---------------------------------------------------------------------------
_DEF_RE = re.compile(r'^[ \t]*(?:def|entry)\s+(\w+)\b', re.MULTILINE)

def function_spans(fut_text):
    matches = list(_DEF_RE.finditer(fut_text))
    spans = []
    for i, m in enumerate(matches):
        start = fut_text.count('\n', 0, m.start()) + 1
        end = (fut_text.count('\n', 0, matches[i+1].start()) + 1
               if i + 1 < len(matches) else 10**9)
        spans.append((start, end, m.group(1)))
    return spans

_FUT_FILES = [
    'autotrade/equilibrium.fut',
    'autotrade/trmodel.fut',
    'run_equilibrium.fut',
    'run_equilibrium_man.fut',
    'run_equilibrium_full_ad.fut',
    'run_equilibrium_almost_full_ad.fut',
    'run_equilibrium_module.fut',
    'lib/github.com/diku-dk/linalg/dpsolve.fut',
    'lib/github.com/diku-dk/linalg/lu.fut',
    'lib/github.com/diku-dk/linalg/lup.fut',
]
FN_MAPS = {}
for rel in _FUT_FILES:
    p = FUT2 / rel
    if p.exists():
        FN_MAPS[rel] = function_spans(p.read_text())
print('FN_MAPS:', {k: len(v) for k, v in FN_MAPS.items()})

_SEG_RE = re.compile(r'^([^:]+):(\d+):')

def segment_file_and_fn(seg):
    """Return (file, fn) for a `file:line:cols` segment. fn is None if the
    line doesn't fall inside any recognised def/entry."""
    m = _SEG_RE.match(seg.strip())
    if not m:
        return None, None
    file, line = m.group(1), int(m.group(2))
    spans = FN_MAPS.get(file)
    if spans is None:
        return file, None
    for s, e, name in spans:
        if s <= line < e:
            return file, name
    return file, None

# ---------------------------------------------------------------------------
# Function-set bookkeeping for bucket dispatch.
# ---------------------------------------------------------------------------
DISPATCHER_FNS = {  # ed_ded_price_all variants: presence => inside a Newton step
    'ed_ded_price_all',
    'ed_ded_price_all_sa',
    'ed_ded_price_all_full_ad',
    'ed_ded_price_all_full_ad_alt',
    'ed_ded_price_all_almost_full_ad',
    'ed_ded_price_all_man',
    'ed_ded_price_all_man_sa',
    'ed_ded_price_all_dctp_only',
    'ed_ded_price_all_dv_dccp',
}
DERIVATIVE_FNS = {  # bucket = ded
    # utility / scrap / bellman / V / EV
    'utility_dprice_man', 'utility_dprice_ad',
    'ccp_scrap_dprice_man', 'ccp_scrap_dprice_ad',
    'dbellman_prices_man', 'dbellman_prices_ad',
    'dev_fixed_point_dprice',
    'dv_dprice_man', 'dv_dprice_dev',
    # ccp / ctp derivatives
    'dccp_dprices_from_du_dev', 'dccp_dprices_from_dev_dv',
    'dccp_dprice_man',
    'ccp_dprice_full',
    'dctp_dprices_from_du_dev', 'dctp_dprices_from_dccp',
    'dctp_dprice_man',
    'ctp_dprice_full',
    # ded itself (per-tau and combined)
    'ded_dprice_ad', 'ded_dprice_man',
    'ded_dprice_tau',
    'ded_dprice_tau_man',
    'ded_dprice_tau_full_ad', 'ded_dprice_tau_full_ad_alt',
    'ded_dprice_tau_almost_full_ad',
    'ded_dprice_tau_dctp_only', 'ded_dprice_tau_dv_dccp',
    # ergodic+derivative (`lup` linear solve over q and dq jointly)
    'ergodic_with_dprice',
}
ED_FNS = {  # bucket = ed
    'ergodic',
    'demand_supply_tau', 'demand_supply_all',
    'ed_tau', 'ed_tau_from_ccp',
}
# bucket = inner_dp_per_newton: detected by *file visit*, not by function set.
# Any chain segment whose file is dpsolve.fut counts as inner DP — unless the
# chain also went through spp_price_solve (init phase), which the in_init
# branch handles first.
DPSOLVE_FILE = 'lib/github.com/diku-dk/linalg/dpsolve.fut'

INIT_FNS   = {'spp_price_solve'}
NEWTON_FNS = {'newton', 'newton_step'}

# Hand-classified <unknown:…> kernels. These are multicore profile-event
# names whose generated C body has no source-location chain (lmad_copy /
# task-scheduler wrappers), so the by-source.txt reports them as <unknown>.
# Identified by reading the .c bodies in saved_c_files_for_local_profiling/
# and tracing back through the calling segred parloop's add_event chain.
# Format: kernel_name -> (phase, bucket, function).
#
# All entries below are the row permute (perm.permute p mat at lup.fut:69)
# inside lup.lup's `step` loop, called from `ergodic_with_dprice ctp dctp`
# at the end of a ded_dprice_tau* function. Same logical work; the IDs
# differ because the multicore backend emits one kernel per fused dispatcher
# variant.
_ERG_DP = ('newton', 'ded', 'ergodic_with_dprice')
KERNEL_OVERRIDES = {
    # run_equilibrium_man.c:  ded_dprice_tau_man (eq.fut:581) -> ergodic_with_dprice
    'futhark_mc_segmap_parloop_514214':       _ERG_DP,
    'futhark_mc_segmap_parloop_514214_total': _ERG_DP,
    'futhark_mc_segmap_task_514212':          _ERG_DP,
    'futhark_mc_segmap_parloop_513595':       _ERG_DP,
    'futhark_mc_segmap_parloop_513595_total': _ERG_DP,
    'futhark_mc_segmap_task_513593':          _ERG_DP,
    # run_equilibrium.c:  ded_dprice_tau (eq.fut:562) -> ergodic_with_dprice
    'futhark_mc_segmap_parloop_544339':       _ERG_DP,
    'futhark_mc_segmap_parloop_544339_total': _ERG_DP,
    'futhark_mc_segmap_task_544337':          _ERG_DP,
    'futhark_mc_segmap_parloop_544413':       _ERG_DP,
    'futhark_mc_segmap_parloop_544413_total': _ERG_DP,
    'futhark_mc_segmap_task_544411':          _ERG_DP,
}

# Hand-classified chain-end overrides. When the deepest segment matches one
# of these `file:line:cols` strings, force the classification. Used for
# kernels whose deepest source span is misleadingly tight (a single fused
# multiply / index op) but whose body actually does substantial upstream
# work that semantically belongs to a different bucket.
CHAIN_END_OVERRIDES = {
    # Fused per-tau derivative+ed assembly in ed_ded_price_all body. Deepest
    # span is the `y * w` tw multiply in `edfs_scaled`, but the kernel
    # actually writes the per-tau dctp/dccp tensor block, runs the ed_tau
    # ns-reduction, and then multiplies by tw at the end. Mostly ded work.
    'autotrade/equilibrium.fut:679:60-65': ('newton', 'ded', 'ded_dprice_tau'),      # default
    'autotrade/equilibrium.fut:884:60-65': ('newton', 'ded', 'ded_dprice_tau_man'),  # manual
}

# Most-specific to least-specific. First match in chain wins. The
# ed_ded_price_all_* family sits near the bottom so it only surfaces as the
# function label when no narrower function appears in the chain (i.e., the
# dispatch-body case).
PRECEDENCE = [
    'dev_fixed_point_dprice',
    'dv_dprice_man', 'dv_dprice_dev',
    'dccp_dprices_from_du_dev', 'dccp_dprices_from_dev_dv',
    'dccp_dprice_man',
    'dctp_dprices_from_du_dev', 'dctp_dprices_from_dccp',
    'dctp_dprice_man',
    'dbellman_prices_man', 'dbellman_prices_ad',
    'utility_dprice_man', 'utility_dprice_ad',
    'ccp_scrap_dprice_man', 'ccp_scrap_dprice_ad',
    'ergodic_with_dprice',
    'ccp_dprice_full', 'ctp_dprice_full',
    'ded_dprice_tau_man',
    'ded_dprice_tau_full_ad', 'ded_dprice_tau_full_ad_alt',
    'ded_dprice_tau_almost_full_ad',
    'ded_dprice_tau_dctp_only', 'ded_dprice_tau_dv_dccp',
    'ded_dprice_tau',
    'ded_dprice_ad', 'ded_dprice_man',
    'ergodic',
    'demand_supply_tau', 'demand_supply_all',
    'ed_tau', 'ed_tau_from_ccp',
    # Inner DP per Newton step (dpsolve.fut algorithms)
    'nk', 'sa', 'poly',
    # spp_price_solve internals (init-phase only)
    'bellman_spp_with_deriv_p', 'bellman_spp_with_deriv',
    'solve_spp_single',
    'spp_price_solve',
    # Dispatch-body fallbacks (when nothing narrower matches)
    'ed_ded_price_all_man_sa', 'ed_ded_price_all_man',
    'ed_ded_price_all_full_ad_alt', 'ed_ded_price_all_full_ad',
    'ed_ded_price_all_almost_full_ad',
    'ed_ded_price_all_dctp_only', 'ed_ded_price_all_dv_dccp',
    'ed_ded_price_all_sa', 'ed_ded_price_all',
    # Last-resort Newton fallbacks
    'newton_step', 'newton',
]

def classify(full_chain):
    """Return (phase, bucket, function)."""
    if not full_chain:
        return ('other', 'other', '?')
    if full_chain.startswith('<unknown:'):
        kernel_name = full_chain[len('<unknown:'):].rstrip('>')
        if kernel_name in KERNEL_OVERRIDES:
            return KERNEL_OVERRIDES[kernel_name]
        return ('other', 'other', '<unknown>')

    # Chain-end override: force classification when the deepest segment
    # (the kernel's profiler-attributed location) matches a known fused
    # kernel whose deepest AST node misrepresents the bulk of the work.
    segments = full_chain.split('->')
    deepest = segments[-1].strip()
    if deepest in CHAIN_END_OVERRIDES:
        return CHAIN_END_OVERRIDES[deepest]

    fns = set()
    files = set()
    for seg in segments:
        file, fn = segment_file_and_fn(seg)
        if file:
            files.add(file)
        if fn:
            fns.add(fn)
    if not fns:
        return ('other', 'other', '?')

    in_init    = bool(fns & INIT_FNS)
    in_ed_ded  = bool(fns & DISPATCHER_FNS)
    in_newton  = bool(fns & NEWTON_FNS) or in_ed_ded
    in_dpsolve = DPSOLVE_FILE in files

    if in_init and not in_newton:
        # Any dpsolve traffic that gets here came via spp — stays in init.
        phase, bucket = 'init', 'init_dp'
    elif in_ed_ded:
        phase = 'newton'
        if in_dpsolve:
            # Inner DP per Newton step — dps.poly/sa/nk inside sol_evs.
            bucket = 'inner_dp_per_newton'
        elif fns & DERIVATIVE_FNS:
            bucket = 'ded'
        elif fns & ED_FNS:
            bucket = 'ed'
        else:
            bucket = 'newton_dispatch'
    elif in_dpsolve:
        # dpsolve visited but no dispatcher in chain — count as inner DP work
        # in the Newton phase (only place dpsolve is called outside init).
        phase, bucket = 'newton', 'inner_dp_per_newton'
    elif in_newton:
        phase, bucket = 'newton', 'newton_linsolve'
    else:
        phase, bucket = 'other', 'other'

    fn_label = '?'
    for cand in PRECEDENCE:
        if cand in fns:
            fn_label = cand
            break
    return phase, bucket, fn_label

# ---------------------------------------------------------------------------
# Build the by-source DataFrame, augmented with phase/bucket/function.
# ---------------------------------------------------------------------------
def parse_by_source(path):
    rows = []
    with open(path) as f:
        for line in f:
            if line.startswith('source') or line.startswith('-'):
                continue
            line = line.rstrip()
            if not line:
                continue
            parts = line.rsplit(None, 4)
            if len(parts) != 5:
                continue
            src, nk, ct, ms, fr = parts
            try:
                nk_i, ct_i, ms_f, fr_f = int(nk), int(ct), float(ms), float(fr)
            except ValueError:
                continue
            if src.startswith('<unknown:'):
                kernel_name = src[len('<unknown:'):].rstrip('>')
                rows.append({
                    'deepest_file':  '<unknown>',
                    'deepest_lines': kernel_name,
                    'call_path':     '<unknown>',
                    '#kernels':      nk_i,
                    'count':         ct_i,
                    'sum_ms':        ms_f,
                    'frac':          fr_f,
                    'full_chain':    src,
                })
                continue
            segments = src.split('->')
            deepest_file, deepest_lines = split_chain_segment(segments[-1])
            rows.append({
                'deepest_file':  deepest_file,
                'deepest_lines': deepest_lines,
                'call_path':     formatted_call_path(segments),
                '#kernels':      nk_i,
                'count':         ct_i,
                'sum_ms':        ms_f,
                'frac':          fr_f,
                'full_chain':    src,
            })
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    cls = df['full_chain'].apply(classify)
    df['phase']    = [c[0] for c in cls]
    df['bucket']   = [c[1] for c in cls]
    df['function'] = [c[2] for c in cls]
    return df.sort_values('sum_ms', ascending=False).reset_index(drop=True)

# ---------------------------------------------------------------------------
# Hierarchical subtotal table: phase -> bucket -> function. Each level's row
# shows the sum of every row beneath it (so a narrow function contributes to
# its bucket subtotal AND its phase subtotal). Fractions are summed from each
# row's by-source.txt `frac`, which is fraction of true total runtime — so
# subtotals can sum to < 1.0 when some runtime is unattributed.
# ---------------------------------------------------------------------------
def hierarchical_summary(df):
    if df.empty:
        return pd.DataFrame()
    rows = []
    phase_order = (df.groupby('phase')['sum_ms'].sum()
                     .sort_values(ascending=False).index)
    for phase in phase_order:
        pdf = df[df['phase'] == phase]
        rows.append({'phase': phase, 'bucket': '', 'function': '',
                     'sum_ms': pdf['sum_ms'].sum(),
                     'count':  int(pdf['count'].sum()),
                     'frac':   pdf['frac'].sum()})
        bucket_order = (pdf.groupby('bucket')['sum_ms'].sum()
                           .sort_values(ascending=False).index)
        for bucket in bucket_order:
            bdf = pdf[pdf['bucket'] == bucket]
            rows.append({'phase': '', 'bucket': bucket, 'function': '',
                         'sum_ms': bdf['sum_ms'].sum(),
                         'count':  int(bdf['count'].sum()),
                         'frac':   bdf['frac'].sum()})
            fn_order = (bdf.groupby('function')['sum_ms'].sum()
                           .sort_values(ascending=False).index)
            for fn in fn_order:
                fdf = bdf[bdf['function'] == fn]
                rows.append({'phase': '', 'bucket': '', 'function': fn,
                             'sum_ms': fdf['sum_ms'].sum(),
                             'count':  int(fdf['count'].sum()),
                             'frac':   fdf['frac'].sum()})
    return pd.DataFrame(rows)

In [10]:
files = sorted(HERE.glob('*.by-source.txt'))
if not files:
    print('No *.by-source.txt files in', HERE)

for p in files:
    display(Markdown(f'### `{p.name}`'))
    df = parse_by_source(p)
    if df.empty:
        display(Markdown('_(no data rows)_'))
        continue
    display(Markdown('**Hierarchical subtotals (phase → bucket → function)**'))
    display(hierarchical_summary(df))
    display(Markdown('**Per-source rows**'))
    display(df.drop(columns=['full_chain']))

### `1_spec-run_equilibrium-local-multicore.by-source.txt`

**Hierarchical subtotals (phase → bucket → function)**

,phase,bucket,function,sum_ms,count,frac
0,newton_inner,,,71390.07,57334,0.964647
1,,newton_step_other,,64941.31,8767,0.877509
2,,,newton_step,64941.31,8767,0.877509
3,,ded,,4432.11,31037,0.059888
4,,,ergodic_with_dprice,4061.34,29861,0.054878
5,,,dev_fixed_point_dprice,370.77,1176,0.005010
6,,ed,,2016.65,17530,0.027250
7,,,ergodic,1945.10,17278,0.026283
8,,,demand_supply_tau,71.55,252,0.000967
9,other,,,1502.77,15873,0.020306


**Per-source rows**

,deepest_file,deepest_lines,call_path,#kernels,count,sum_ms,frac,phase,bucket,function
0,autotrade/equilibrium.fut,679:60-65,run_equilibrium.fut:110:7-112:69 → run_equilibrium_module.fut:59:34-66:142 → run_equilibrium_module.fut:35:8-38:35 → run_equilibrium_module.fut:28:8-32:45 → run_equilibrium.fut:15:26-46 → equilibrium.fut:679:60-65,3,252,40467.25,0.5329,newton_inner,newton_step_other,newton_step
1,autotrade/equilibrium.fut,551:60-679:122,run_equilibrium.fut:110:7-112:69 → run_equilibrium_module.fut:59:34-66:142 → run_equilibrium_module.fut:35:8-38:35 → run_equilibrium_module.fut:28:8-32:45 → run_equilibrium.fut:15:26-46 → equilibrium.fut:551:60-679:122,9,150,13024.92,0.1714,newton_inner,newton_step_other,newton_step
2,autotrade/equilibrium.fut,551:60-679:70,run_equilibrium.fut:110:7-112:69 → run_equilibrium_module.fut:59:34-66:142 → run_equilibrium_module.fut:35:8-38:35 → run_equilibrium_module.fut:28:8-32:45 → run_equilibrium.fut:15:26-46 → equilibrium.fut:551:60-679:70,3,42,8237.85,0.1086,newton_inner,newton_step_other,newton_step
3,lib/github.com/diku-dk/linalg/lup.fut,73:7-75:77,run_equilibrium.fut:110:7-112:69 → run_equilibrium_module.fut:59:34-66:142 → run_equilibrium_module.fut:35:8-38:35 → run_equilibrium_module.fut:28:8-32:45 → run_equilibrium.fut:15:26-46 → equilibrium.fut:646:46-661:44 → equilibrium.fut:550:61-562:61 → equilibrium.fut:481:23-39 → lup.fut:83:51-63 → lup.fut:73:7-75:77,9,21766,3668.14,0.0482,newton_inner,ded,ergodic_with_dprice
4,lib/github.com/diku-dk/linalg/lup.fut,73:7-75:77,run_equilibrium.fut:110:7-112:69 → run_equilibrium_module.fut:59:34-66:142 → run_equilibrium_module.fut:35:8-38:35 → run_equilibrium_module.fut:28:8-32:45 → run_equilibrium.fut:15:26-46 → equilibrium.fut:675:33-44 → equilibrium.fut:131:20-34 → lup.fut:106:18-23 → lup.fut:83:51-63 → lup.fut:73:7-75:77,9,17278,1945.10,0.0256,newton_inner,ed,ergodic
5,lib/github.com/diku-dk/linalg/lu.fut,118:32-73,run_equilibrium.fut:110:7-112:69 → run_equilibrium_module.fut:59:34-66:142 → run_equilibrium_module.fut:35:8-38:35 → run_equilibrium_module.fut:28:8-32:45 → run_equilibrium.fut:15:26-46 → equilibrium.fut:647:42-658:58 → equilibrium.fut:646:66-654:123 → dpsolve.fut:228:28-238:57 → dpsolve.fut:206:32-63 → dpsolve.fut:257:48-64 → lu.fut:183:17-31 → lu.fut:157:13-28 → lu.fut:118:32-73,3,1368,975.14,0.0128,newton_inner,newton_step_other,newton_step
6,lib/github.com/diku-dk/linalg/lu.fut,117:5-118:74,run_equilibrium.fut:110:7-112:69 → run_equilibrium_module.fut:59:34-66:142 → run_equilibrium_module.fut:35:8-38:35 → run_equilibrium_module.fut:28:8-32:45 → run_equilibrium.fut:15:26-46 → equilibrium.fut:647:42-658:58 → equilibrium.fut:646:66-654:123 → dpsolve.fut:228:28-238:57 → dpsolve.fut:206:32-63 → dpsolve.fut:257:48-64 → lu.fut:183:17-31 → lu.fut:157:13-28 → lu.fut:117:5-118:74,6,228,917.31,0.0120,newton_inner,newton_step_other,newton_step
7,autotrade/equilibrium.fut,81:101-84:99,equilibrium.fut:90:28-94:43 → equilibrium.fut:89:45-94:43 → equilibrium.fut:81:101-84:99,3,4,716.37,0.0095,init,init_dp,solve_spp_single
8,lib/github.com/diku-dk/linalg/lu.fut,44:10-58:16,run_equilibrium.fut:110:7-112:69 → run_equilibrium_module.fut:59:34-66:142 → run_equilibrium_module.fut:35:8-38:35 → run_equilibrium_module.fut:28:8-32:45 → run_equilibrium.fut:15:26-46 → equilibrium.fut:647:42-658:58 → equilibrium.fut:646:66-654:123 → dpsolve.fut:228:28-238:57 → dpsolve.fut:206:32-63 → dpsolve.fut:257:48-64 → lu.fut:183:17-31 → lu.fut:157:13-28 → lu.fut:122:18-46 → lu.fut:44:10-58:16,6,148,432.84,0.0057,newton_inner,newton_step_other,newton_step
9,autotrade/equilibrium.fut,37:79-61:50,equilibrium.fut:90:28-97:17 → equilibrium.fut:89:45-96:71 → equilibrium.fut:37:79-61:50,6,10,397.21,0.0053,init,init_dp,bellman_spp_with_deriv_p


...

### `5_spec-run_equilibrium_man-local-multicore.by-source.txt`

**Hierarchical subtotals (phase → bucket → function)**

,phase,bucket,function,sum_ms,count,frac
0,newton_inner,,,274071.94,171702,0.576392
1,,ded,,217647.63,86201,0.457728
2,,,ded_dprice_tau_man,119200.92,1524,0.250688
3,,,ergodic_with_dprice,74917.36,36677,0.157556
4,,,dev_fixed_point_dprice,23529.35,48000,0.049484
5,,newton_step_other,,48217.98,50244,0.101406
6,,,newton_step,48217.98,50244,0.101406
7,,ed,,8206.33,35257,0.017258
8,,,ergodic,8206.33,35257,0.017258
9,other,,,189276.94,54864,0.398062


**Per-source rows**

,deepest_file,deepest_lines,call_path,#kernels,count,sum_ms,frac,phase,bucket,function
0,<unknown>,futhark_mc_segmap_parloop_514214,<unknown>,1,24384,148903.10,0.2997,other,other,<unknown>
1,autotrade/equilibrium.fut,568:60-884:122,run_equilibrium_man.fut:109:7-111:69 → run_equilibrium_module.fut:59:34-66:142 → run_equilibrium_module.fut:35:8-38:35 → run_equilibrium_module.fut:28:8-32:45 → run_equilibrium_man.fut:14:26-50 → equilibrium.fut:568:60-884:122,9,276,83444.94,0.1679,newton_inner,ded,ded_dprice_tau_man
2,lib/github.com/diku-dk/linalg/lup.fut,73:7-75:77,run_equilibrium_man.fut:109:7-111:69 → run_equilibrium_module.fut:59:34-66:142 → run_equilibrium_module.fut:35:8-38:35 → run_equilibrium_module.fut:28:8-32:45 → run_equilibrium_man.fut:14:26-50 → equilibrium.fut:851:50-866:48 → equilibrium.fut:567:65-581:61 → equilibrium.fut:481:23-39 → lup.fut:83:51-63 → lup.fut:73:7-75:77,9,35513,43656.34,0.0879,newton_inner,ded,ergodic_with_dprice
3,autotrade/equilibrium.fut,479:38-488:41,run_equilibrium_man.fut:109:7-111:69 → run_equilibrium_module.fut:59:34-66:142 → run_equilibrium_module.fut:35:8-38:35 → run_equilibrium_module.fut:28:8-32:45 → run_equilibrium_man.fut:14:26-50 → equilibrium.fut:851:50-866:48 → equilibrium.fut:567:65-581:61 → equilibrium.fut:479:38-488:41,3,1080,23235.16,0.0468,newton_inner,ded,ergodic_with_dprice
4,autotrade/equilibrium.fut,884:60-65,run_equilibrium_man.fut:109:7-111:69 → run_equilibrium_module.fut:59:34-66:142 → run_equilibrium_module.fut:35:8-38:35 → run_equilibrium_module.fut:28:8-32:45 → run_equilibrium_man.fut:14:26-50 → equilibrium.fut:884:60-65,3,1080,22554.44,0.0454,newton_inner,newton_step_other,newton_step
5,autotrade/equilibrium.fut,402:13-408:43,run_equilibrium_man.fut:109:7-111:69 → run_equilibrium_module.fut:59:34-66:142 → run_equilibrium_module.fut:35:8-38:35 → run_equilibrium_module.fut:28:8-32:45 → run_equilibrium_man.fut:14:26-50 → equilibrium.fut:851:50-866:48 → equilibrium.fut:567:48-579:49 → equilibrium.fut:402:13-408:43,3,1080,19781.47,0.0397,newton_inner,ded,ded_dprice_tau_man
6,<unknown>,futhark_mc_segmap_task_514212,<unknown>,1,1524,14498.06,0.0292,other,other,<unknown>
7,<unknown>,futhark_mc_segmap_parloop_514214_total,<unknown>,1,1524,14446.49,0.0291,other,other,<unknown>
8,autotrade/equilibrium.fut,568:60-884:70,run_equilibrium_man.fut:109:7-111:69 → run_equilibrium_module.fut:59:34-66:142 → run_equilibrium_module.fut:35:8-38:35 → run_equilibrium_module.fut:28:8-32:45 → run_equilibrium_man.fut:14:26-50 → equilibrium.fut:568:60-884:70,3,84,9182.66,0.0185,newton_inner,ded,ded_dprice_tau_man
9,lib/github.com/diku-dk/linalg/lup.fut,73:7-75:77,run_equilibrium_man.fut:109:7-111:69 → run_equilibrium_module.fut:59:34-66:142 → run_equilibrium_module.fut:35:8-38:35 → run_equilibrium_module.fut:28:8-32:45 → run_equilibrium_man.fut:14:26-50 → equilibrium.fut:880:33-44 → equilibrium.fut:131:20-34 → lup.fut:106:18-23 → lup.fut:83:51-63 → lup.fut:73:7-75:77,9,35257,8206.33,0.0166,newton_inner,ed,ergodic
